# 🧬 Your First Genomics Agent with Strands

In this notebook you build your very first AI agent, in code, using the Strands Agents framework. You will start with a plain language model that can chat but knows nothing about your data, then turn it into a genomics Research Assistant that can answer questions directly from your own variant results.

## 🎯 What You'll Learn

1. **What an agent is** - model, tools, and prompt working together
2. **The agentic loop** - how an agent observes, plans, acts, and reflects
3. **The @tool decorator** - how to give an agent a new capability in a few lines of Python
4. **Why tools matter** - the difference between a chatbot and an agent that can reach your data

## 🏗️ What You'll Build

1. **A basic agent** - a model with no tools, to see its limits
2. **Your Research Assistant** - the same model plus one custom tool that reads your variant data
3. **A working query** - ask your variant results a question in plain language and get a real answer

If you want a no-code first impression before writing any code, revisit the Bedrock Playground walkthrough on the :link[Navigating your AWS account]{href="../navigating-your-aws-account/"} page.

---

## 🛠️ Step 1: Environment Setup

First, let's install the required dependencies and set up our environment.

In [9]:
from IPython.display import HTML

HTML("""
<style>
/* Make all JupyterLab output areas scrollable */
.jp-OutputArea-output {
    max-height: 300px;
    overflow-y: auto;
}
</style>
""")

In [10]:
# Install required dependencies
# Note: You may see dependency conflict warnings from pre-installed SageMaker packages
# (e.g., sparkmagic, shap). These are safe to ignore - they do not affect this workshop.
!pip install "pandas>=2.0,<3.0" --upgrade --quiet
!pip install strands-agents boto3 uv --quiet

# Set up environment for tool consent bypass (for automated workflows)
import os
os.environ["BYPASS_TOOL_CONSENT"] = "true"

print("✅ Environment setup complete!")

✅ Environment setup complete!


## 🧱 Step 2: What Is an Agent?

Before we write any code, let's be clear about what an agent actually is. An agent is made of three simple parts:

1. **A model** - the language model that provides the reasoning and language skills (like Claude on Amazon Bedrock)
2. **Tools** - functions the agent can call to do real work, such as reading your variant file or checking a workflow
3. **A prompt** - the system prompt that tells the agent its role and how to behave

Put simply: **agent = model + tools + prompt**.

When you ask an agent a question, it runs a short cycle called the **agentic loop**:

- **Observe** - read your question and the information it already has
- **Plan** - decide whether it can answer directly or whether it needs a tool
- **Act** - call a tool if one is needed (for example, look up variants in a gene)
- **Reflect** - read the tool result and turn it into a plain language answer

The key idea is that a model on its own can only talk. Tools are what let an agent reach into your data and take action. Let's see that difference for ourselves.

## 📚 What Is Strands Agents?

Now that you know what an agent is, here is the framework you will use to build one. Strands Agents is a Python framework for building AI agents that can:

- Use tools to interact with external systems
- Maintain conversation context and memory
- Work with multiple LLM providers (Bedrock, OpenAI, and others)
- Integrate with protocols like MCP (Model Context Protocol)
- Deploy to various environments (standalone, container, Bedrock AgentCore, and others)

### Key Components

1. **Agent** - the core AI assistant with a specific role and capabilities
2. **Model** - the underlying LLM (for example Claude on Amazon Bedrock)
3. **Tools** - functions the agent can call to perform actions
4. **System Prompt** - instructions that define the agent's behavior

In this module you build the simplest possible version: a model plus one tool plus a system prompt. Let's start.

In [13]:
from strands import Agent
from strands.models import BedrockModel

# Create a basic agent with no tools
basic_agent = Agent(
    name="Basic Genomics Assistant",
    description="A simple assistant with no tools",
    model=BedrockModel(
        model_id="global.anthropic.claude-opus-5-5",
        # 💡 Try Opus 5.5 for deeper reasoning (slower, higher cost): "global.anthropic.claude-opus-5-5"
    )
)

# A general knowledge question - the model can answer this on its own
print("🧪 General question: What is a somatic variant?")
print("=" * 60)
response = basic_agent("What is a somatic variant?")
print(response)

# A data-specific question - the model has NO access to your variant data yet
print("\n\n🧪 Data-specific question: What variants are in TP53 in my sample?")
print("=" * 60)
response = basic_agent("What variants are in TP53 in my sample?")
print(response)

print("\n👉 Notice: the basic agent has no tools, so it cannot see your variant data.")
print("   It can explain concepts, but it cannot answer questions about YOUR sample yet.")

🧪 General question: What is a somatic variant?
A **somatic variant** (also called a somatic mutation) is a change in DNA that occurs in a **non-reproductive (somatic) cell** at some point after conception. Because it arises in body cells rather than in eggs or sperm, it is **not inherited from parents and is not passed on to children**.

## Key Features

- **Acquired, not inherited:** It arises during a person's lifetime, often from DNA replication errors or environmental damage such as UV light, tobacco smoke, radiation, or certain chemicals.
- **Present in only some cells:** It is found only in the cell where it arose and that cell's descendants. The rest of the body doesn't carry it.
- **Mosaicism:** When a variant arises early in development, it can be present in a large fraction of cells across one or more tissues. A person whose cells differ genetically in this way is called *mosaic*.

## Contrast with Germline Variants

| | **Somatic variant** | **Germline variant** |
|---|---|-

## 🔬 Step 3: Give Your Agent Access to Your Data

The basic agent could explain what a somatic variant is, but it could not tell you what is in your own sample. That is because it has no way to reach your data.

To fix that, we first need to load your variant data into the notebook. Your data comes from the pre-run Mutect2 pipeline, which produced a MAF (Mutation Annotation Format) file. A MAF file is a table where each row is one variant found in the sample, with columns for the gene, the type of change, its position, and more.

The cell below finds that MAF file in your account and loads it into a table we can work with.

In [4]:
import boto3
import pandas as pd
from io import StringIO

# Get the results bucket and find the MAF file
s3 = boto3.client('s3')
omics = boto3.client('omics')
cfn = boto3.client('cloudformation')

# Get bucket name from CloudFormation stack
stack = cfn.describe_stacks(StackName='infrastructure')['Stacks'][0]
results_bucket = [o['OutputValue'] for o in stack['Outputs'] if o['OutputKey'] == 'WorkflowResultsBucket'][0]

# Find the MAF file from the completed workflow run
runs = omics.list_runs()
for run in runs['items']:
    if run.get('status') == 'COMPLETED':
        run_id = run['id']
        response = s3.list_objects_v2(Bucket=results_bucket, Prefix=f"omics_output/{run_id}/")
        for obj in response.get('Contents', []):
            if obj['Key'].endswith('.maf'):
                maf_key = obj['Key']
                break
        break

obj = s3.get_object(Bucket=results_bucket, Key=maf_key)
maf_df = pd.read_csv(StringIO(obj['Body'].read().decode('utf-8')), sep='\t', comment='#')
print(f"✅ Loaded {len(maf_df)} variants from {maf_key}")
print(maf_df[['Hugo_Symbol', 'Variant_Classification', 'HGVSp_Short']].head(10))


✅ Loaded 41 variants from omics_output/5820471/out/maf_file/output.maf
  Hugo_Symbol Variant_Classification HGVSp_Short
0        UBR4      Missense_Mutation    p.E2097K
1        MTF1      Missense_Mutation     p.S508P
2      LINGO4      Missense_Mutation     p.D513N
3      CCDC93      Missense_Mutation     p.P440S
4        CPS1      Missense_Mutation     p.T166I
5       PARP3      Missense_Mutation     p.E200Q
6       PRDM9                 Silent     p.D201=
7      TRIM39                 Silent     p.R374=
8     HLA-DRA      Missense_Mutation     p.Y175C
9        HEY2      Missense_Mutation     p.A247T


## 🔧 Step 4: Add One Tool with the @tool Decorator

Now we can connect the agent to your data. In Strands, you turn any Python function into an agent tool by adding the `@tool` decorator above it. That is all it takes.

The agent reads the function's name, its description (the docstring), and its inputs, and decides on its own when to call it. You do not have to tell the agent step by step what to do. When a question needs the tool, the agent calls it, reads the result, and writes a plain language answer.

We will add exactly **one** tool right now to keep your first agent simple: `query_variants_by_gene`, which looks up all variants in a gene you name. In Module 2 you will grow this toolkit with more tools and connect an entire platform of tools through MCP.

In [5]:
from strands import tool

@tool
def query_variants_by_gene(gene_symbol: str) -> str:
    """Query the somatic variant dataset for all variants in a given gene."""
    results = maf_df[maf_df['Hugo_Symbol'].str.upper() == gene_symbol.upper()]
    if results.empty:
        return f"No variants found for gene {gene_symbol} in this sample."
    cols = ['Hugo_Symbol', 'Variant_Classification', 'HGVSp_Short', 'Chromosome', 'Start_Position', 't_depth', 't_alt_count']
    available_cols = [c for c in cols if c in results.columns]
    summary = results[available_cols].to_string(index=False)
    return f"Found {len(results)} variant(s) in {gene_symbol}:\n{summary}"

print("✅ One tool defined: query_variants_by_gene")


✅ One tool defined: query_variants_by_gene


## 🤖 Step 5: Create Your Research Assistant

Now we bring the three parts together into a real agent:

- **model** - the same Claude model on Bedrock we used before
- **tools** - the single `query_variants_by_gene` tool we just defined
- **system_prompt** - a short instruction that gives the agent its role as a genomics research assistant

The system prompt is how you specialize an agent. It is the main difference between the bare chatbot from Step 2 and the Research Assistant we are about to build. Let's create the agent and then re-ask the exact same data-specific question that the basic agent could not answer.

In [6]:
# Create the Research Assistant - a model, one tool, and a system prompt
research_assistant = Agent(
    name="Genomics Research Assistant",
    model=BedrockModel(
        model_id="global.anthropic.claude-sonnet-5.5",
        # 💡 Try Opus 5.5 for deeper reasoning (slower, higher cost): "global.anthropic.claude-opus-5-5"
    ),
    tools=[query_variants_by_gene],
    system_prompt="""You are a genomics research assistant helping researchers interpret somatic variant data from cancer samples.
Use the available tools to answer questions about variants. Always explain findings in plain language."""
)

# Re-ask the SAME data-specific question the basic agent could not answer
print("🧪 Data-specific question: What variants are in TP53 in my sample?")
print("=" * 60)
response = research_assistant("What variants are in TP53 in my sample?")
print(response)
print(f"\n📊 Tools used: {list(response.metrics.tool_metrics.keys())}")


🧪 Data-specific question: What variants are in TP53 in my sample?

Tool #1: query_variants_by_gene
## TP53 Variant Found in Your Sample

I found **1 variant** in the TP53 gene:

| Feature | Detail |
|---|---|
| **Variant Type** | Nonsense mutation |
| **Protein Change** | p.R213* |
| **Location** | Chromosome 17, position 7,674,894 |
| **Read Depth** | 38 total reads at this position |
| **Variant Reads** | 20 reads supporting the mutation (~53% allele frequency) |

### What this means in plain language:

- **Nonsense mutation**: This is a mutation that creates a premature "stop" signal in the DNA. Instead of producing a full, functional TP53 protein, the cell makes a truncated (shortened) version that is typically non-functional or quickly degraded.

- **p.R213***: This notation tells us that at amino acid position 213 in the TP53 protein, an arginine (R) has been replaced by a stop codon (indicated by the asterisk *). This means protein synthesis halts at this point, producing a seve

### 💡 What Just Happened?

You asked the exact same question twice.

- In **Step 2**, the basic agent had no tools. It could not see your data, so it could not tell you what variants were in TP53 in your sample.
- In **Step 5**, the Research Assistant had one tool. When you asked the same question, the agent recognized that it needed real data, called `query_variants_by_gene("TP53")`, read the result, and gave you a real, data-backed answer.

That single tool is the whole difference between a chatbot and an agent that works with your data. You can see the agentic loop in the trace above: the agent observed your question, planned to use a tool, acted by calling it, and reflected on the result to write its answer. The `Tools used` line confirms which tool it chose.

---

## 🎓 Success Checklist & Key Takeaways

### ✅ Success Checklist

You have finished Module 1 when you can check all of these:

- [ ] The environment setup cell ran and printed the success message
- [ ] The basic agent answered a general question but could not answer a question about your sample
- [ ] Your MAF variant data loaded and printed a count plus the first few rows
- [ ] You defined the `query_variants_by_gene` tool with the `@tool` decorator
- [ ] The Research Assistant answered the TP53 question using the tool, and the `Tools used` line listed it

### 🔑 Key Takeaways

- An agent is just three parts working together: a model, tools, and a prompt
- A model on its own can only talk. Tools are what let it reach your data and take action
- The `@tool` decorator turns a plain Python function into an agent capability in a few lines
- The agentic loop (observe, plan, act, reflect) is visible in the trace every time the agent runs

### ⚠️ One Tool Is Just the Start

Your Research Assistant can look up variants by gene, but that is all it can do right now. It cannot yet summarize your results, find high-impact variants, or check on your workflow runs. In Module 2 you will grow its toolkit with more tools and connect it to an entire platform of tools through MCP, taking it from one tool to more than 69.